<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/mlp_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Multi-Layer Perceptron from scratch to solve XOR

## Imports

In [ ]:
import math
import random
from typing import Dict, Literal

## Pre-setup

In [ ]:
random.seed(42)

## Sub Layers

In [ ]:
class Linear():
  def __init__(self, w1:None|float=None, w2:None|float=None, b:None|float=None):
    # Model parameters
    self.w1 = w1
    self.w2 = w2
    self.b = b

    if w1 is None:
      self.w1: float =  random.random()
    if w2 is None:
      self.w2: float = random.random()
    if b is None:
      self.b: float = random.random()

    # Internal caches
    self.last_x1: float = None
    self.last_x2: float = None
    self.last_backward: Dict[str, float] = None

  def __call__(self, x1:float, x2:float):
    return self.forward(x1, x2)

  def forward(self, x1:float, x2:float):
    self.last_x1 = x1
    self.last_x2 = x2
    return self.w1 * x1 + self.w2 * x2 + self.b

  def backward(self, prior_grad:float):
    if (self.last_x1 is None) or (self.last_x2 is None):
      raise Exception("First run at least one forward. c:LinearLayer")

    # Model parameter gradients
    self.last_backward = {
        "w1": prior_grad * self.last_x1,
        "w2": prior_grad * self.last_x2,
         "b": prior_grad * 1
    }

    # Input parameter gradients
    input_grad = {
        "x1": prior_grad * self.w1,
        "x2": prior_grad * self.w2
    }

    return self.last_backward, input_grad

  def train(self, learning_rate:float):
    if self.last_backward is None:
        raise Exception("No backward pass has been run. c:LinearLayer")
    self.w1 = self.w1 - learning_rate * self.last_backward["w1"]
    self.w2 = self.w2 - learning_rate * self.last_backward["w2"]
    self.b = self.b - learning_rate * self.last_backward["b"]

In [ ]:
class ReLU():
  def __init__(self):
    self.last_forward: float = None
    self.last_backward: float = None

  def __call__(self, x: float):
    return self.forward(x)

  def forward(self, x: float):
    if x > 0:
      self.last_forward = x
    else:
      self.last_forward = 0
    return self.last_forward

  def backward(self, prior_gradient: float):
    if self.last_forward > 0:
      self.last_backward = prior_gradient * 1
    else:
      self.last_backward = 0
    return self.last_backward

In [ ]:
class Sigmoid():
  def __init__(self):
    # Internal caches
    self.last_forward: float = None
    self.last_backward: float = None

  def __call__(self, x:float):
    return self.forward(x)

  def forward(self, x:float):
    self.last_forward = 1 / (1 + math.exp(-x))
    return self.last_forward

  def backward(self, prior_gradient:float):
    if self.last_forward is None:
      raise Exception("First run at least one forward. c:Sigmoid")

    self.last_backward = prior_gradient * self.last_forward * (1 - self.last_forward)
    return self.last_backward

In [ ]:
class SquaredLoss():
  def __init__(self):
    # Internal cache
    self.last_diff_y_pred_act = None

  def __call__(self, y_pred: float, y_act: float):
    return self.forward(y_pred, y_act)

  def forward(self, y_pred: float, y_act: float):
    self.last_diff_y_pred_act = y_pred - y_act
    return 0.5 * self.last_diff_y_pred_act * self.last_diff_y_pred_act

  def backward(self):
    if self.last_diff_y_pred_act is None:
      raise Exception("First run at least one forward. c:SquaredLoss")

    return self.last_diff_y_pred_act

## XORMLP

In [ ]:
class XORMLP():
  def __init__(self, activ_fn:Literal["relu", "sigmoid"], weight_map:Dict[str, float]|None=None):
    self.l1 = Linear()
    self.l2 = Linear()

    if activ_fn == "relu":
      self.a1 = ReLU()
      self.a2 = ReLU()
    elif activ_fn == "sigmoid":
      self.a1 = Sigmoid()
      self.a2 = Sigmoid()
    else:
      raise Exception("Invalid activ_fn literal provided. c:XORMLP")

    self.o = Linear()

    if weight_map is not None:
      self.l1.w1 = weight_map["w1"]
      self.l1.w2 = weight_map["w2"]
      self.l1.b = weight_map["b1"]

      self.l2.w1 = weight_map["w3"]
      self.l2.w2 = weight_map["w4"]
      self.l2.b = weight_map["b2"]

      self.o.w1 = weight_map["w5"]
      self.o.w2 = weight_map["w6"]
      self.o.b = weight_map["b3"]

  def __call__(self, x1: float, x2: float):
    return self.forward(x1, x2)

  def forward(self, x1:float, x2:float):
    # Hidden layer
    ## First linear/summation layer
    l1_out = self.l1(x1, x2)
    l2_out = self.l2(x1, x2)

    ## Corresponding activations
    a1_out = self.a1(l1_out)
    a2_out = self.a2(l2_out)

    # Output layer
    out = self.o(a1_out, a2_out)
    return out

  def backward(self, loss_grad:float):
    _, o_grad = self.o.backward(loss_grad)

    a1_grad = self.a1.backward(o_grad["x1"])
    a2_grad = self.a2.backward(o_grad["x2"])

    self.l1.backward(a1_grad)
    self.l2.backward(a2_grad)

  def train(self, lr:float):
    self.l1.train(lr)
    self.l2.train(lr)

    self.o.train(lr)


## Validate Forward Propagation

In [ ]:
gold_std_weight_map = {
    "w1": 1,
    "w2": 1,
    "b1": 0,
    "w3": 1,
    "w4": 1,
    "b2": -1,
    "w5": 1,
    "w6": -2,
    "b3": 0
}

In [ ]:
bench_XORMLP = XORMLP(activ_fn="relu", weight_map=gold_std_weight_map)
assert bench_XORMLP(0, 0) == 0
assert bench_XORMLP(0, 1) == 1
assert bench_XORMLP(1, 0) == 1
assert bench_XORMLP(0, 0) == 0

## Training

In [ ]:
X = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

In [ ]:
Y = [0, 1, 1, 0]

In [ ]:
EPOCHS=500
LR = 10e-2

In [ ]:
model = XORMLP(activ_fn="relu")

In [ ]:
loss_fn = SquaredLoss()

In [ ]:
for i in range(EPOCHS):
  loss = 0
  for epoch in range(EPOCHS):
    for (x1, x2), y_act in zip(X, Y):
        y_pred = model(x1, x2)
        loss += loss_fn(y_pred, y_act)
        grad_loss = loss_fn.backward()
        model.backward(grad_loss)
        model.train(LR)

## Validate Model

In [ ]:
assert model(0, 0) < 0.01
assert model(0, 1) > 0.99
assert model(1, 0) > 0.99
assert model(0, 0) < 0.01